In [ ]:
# Geração da base fictícia para modelagem e treinamento do modelo.
import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker('pt_BR')
np.random.seed(42)
random.seed(42)

n = 500

status_list = ['Aprovado', 'Reprovado']
origens = ['LinkedIn', 'Gupy', 'Indicação', 'Catho', 'Indeed', 'Site Empresa']
senioridades = ['Júnior', 'Pleno', 'Sênior']
areas = ['Dados', 'TI', 'Financeiro', 'RH', 'Marketing', 'Operações']
recrutadores = ['Ana', 'Bruno', 'Camila', 'Diego', 'Fernanda']
motivos = [
    'Falta aderencia tecnica',
    'Desistencia candidato',
    'Pretensao salarial alta',
    'Falta experiencia',
    'Fit cultural'
]

dados = []

for i in range(1, n+1):
    data_inscricao = fake.date_between(start_date='-180d', end_date='today')

    status = np.random.choice(status_list, p=[0.58, 0.42])

    if status == 'Aprovado':
        tempo = np.random.randint(5, 15)
        motivo = ''
        teste = 'Aprovado'
    else:
        tempo = np.random.randint(10, 30)
        motivo = random.choice(motivos)
        teste = np.random.choice(['Aprovado', 'Reprovado'], p=[0.35, 0.65])

    data_final = data_inscricao + pd.Timedelta(days=int(tempo))

    senioridade = random.choice(senioridades)

    if senioridade == 'Júnior':
        salario = np.random.randint(3000, 6000)
    elif senioridade == 'Pleno':
        salario = np.random.randint(6000, 10000)
    else:
        salario = np.random.randint(10000, 18000)

    dados.append([
        i,
        status,
        data_inscricao,
        data_final,
        motivo,
        tempo,
        salario,
        random.choice(recrutadores),
        random.choice(origens),
        senioridade,
        random.choice(areas),
        teste
    ])

cols = [
    'ID_Candidato','Status','Data_Inscricao','Data_Finalizacao',
    'Motivo_Reprovacao','Tempo_Processo_Dias','Pretensao_Salarial',
    'Recrutador','Origem_Candidato','Senioridade','Area','Teste_Tecnico'
]

df = pd.DataFrame(dados, columns=cols)

df.to_csv('base_recrutamento_500_linhas.csv', index=False, encoding='utf-8-sig')

print(df.head())
print("\nArquivo salvo: base_recrutamento_500_linhas.csv")

from google.colab import files
files.download('base_recrutamento_500_linhas.csv')

In [ ]:
import pandas as pd

url = "https://docs.google.com/spreadsheets/d/1c8OI3BMQFCczDpqOGrUGt6QaUMJFleXIJPl_-vGQeUw/export?format=csv&gid=1115787433"
df = pd.read_csv(url)
df.info()

#quebra de linha por motivos de organização do código no arquivo.
print("--------------------------------------------------------------------------------------------------------------------")
print(df.head(10))
print(f"\nExistem {df.shape[0]} linhas e {df.shape[1]} colunas no arquivo.")

In [ ]:
dictionary = {}

dict_df = pd.DataFrame(dictionary.items(), columns=['Variável', 'Info'])

# Classificação por tipo
def classificar_tipo(var):
    categoricas = []
    numericas = []
    datas = []

    if var in categoricas:
        return 'Categórica'
    elif var in numericas:
        return 'Numérica'
    elif var in datas:
        return 'Data'
    else:
        return 'Outros'

dict_df['Tipo'] = dict_df['Variável'].apply(classificar_tipo)
dict_df = dict_df.sort_values(['Tipo', 'Variável']).reset_index(drop=True)

display(dict_df)

In [ ]:
print(f"Instâncias: {df.shape[0]} | Atributos: {df.shape[1]}")
print(f"Duplicadas: {df.duplicated().sum()}")

missing_pct = df.isnull().mean() * 100
zeros_pct = (df == 0).mean() * 100
miss_zeros = pd.DataFrame({'feat': df.columns, 'miss_%': missing_pct, 'zero_%': zeros_pct}).sort_values('miss_%', ascending=False)
display(miss_zeros.head(10))

In [ ]:
df['target'] = df['Stage'].apply(lambda x: 'ganha' if 'ganh' in str(x).lower() else ('perdida' if 'perd' in str(x).lower() else 'ganha'))
print("Distribuição Target:")
display(df['target'].value_counts())

ganhas_df = df[df['target']=='ganha']
perdidas_df = df[df['target']=='perdida']

In [ ]:
!pip install ks_2samp
from scipy.stats import ks_2samp

df_ks = df.copy()
df_ks['target'] = np.where(
    df_ks['Stage'].astype(str).str.lower().str.contains('cliente cadastrado'),
    1,
    np.where(
        df_ks['Stage'].astype(str).str.lower().str.contains('oportunidade perdida'),
        0,
        np.nan
    )
)

df_ks = df_ks[df_ks['target'].notna()].copy()
df_ks['target'] = df_ks['target'].astype(int)

print("Distribuição da target:")
print(df_ks['target'].value_counts())
print()
if 'Amount' in df_ks.columns:
    df_ks['Amount'] = (
        df_ks['Amount']
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
    df_ks['Amount'] = pd.to_numeric(df_ks['Amount'], errors='coerce')

numvars = ['Amount', 'Idade da Oportunidade em Dias']
numvars_exist = [col for col in numvars if col in df_ks.columns]

for col in numvars_exist:
    df_ks[col] = pd.to_numeric(df_ks[col], errors='coerce')

print("Colunas usadas:", numvars_exist)
print()

def ks_feature_importance(X, y):
    ks_scores = []

    for col in X.columns:
        grupo0 = X.loc[y == 0, col].dropna()
        grupo1 = X.loc[y == 1, col].dropna()

        if len(grupo0) > 0 and len(grupo1) > 0:
            ks_stat, p_value = ks_2samp(grupo0, grupo1)

            ks_scores.append({
                "feature": col,
                "ks_stat": ks_stat,
                "p_value": p_value
            })

    if len(ks_scores) == 0:
        return pd.DataFrame(columns=["feature", "ks_stat", "p_value"])

    return pd.DataFrame(ks_scores).sort_values(by="ks_stat", ascending=False)

X = df_ks[numvars_exist]
y = df_ks['target']

ks_df = ks_feature_importance(X, y)

if ks_df.empty:
    print("Nenhuma variável conseguiu discriminar")
else:
    display(ks_df)

In [ ]:
# 1) manter apenas colunas de numvars que existem no dataframe
numvars_exist = [col for col in numvars if col in df.columns]
print(f"Colunas encontradas em numvars: {numvars_exist}")

# 2) criar uma cópia para tratar sem bagunçar o df original
df_corr = df.copy()

# 3) tratamento especial para Amount (ex.: '310,00')
if 'Amount' in df_corr.columns:
    df_corr['Amount'] = (
        df_corr['Amount']
        .astype(str)
        .str.replace('.', '', regex=False)   # remove separador de milhar, se houver
        .str.replace(',', '.', regex=False)  # troca vírgula decimal por ponto
        .str.strip()
    )
    df_corr['Amount'] = pd.to_numeric(df_corr['Amount'], errors='coerce')

# 4) converter as demais colunas de numvars para número, se possível
for col in numvars_exist:
    if col != 'Amount':
        df_corr[col] = pd.to_numeric(df_corr[col], errors='coerce')

# 5) identificar quais colunas realmente ficaram numéricas
numvars_validas = df_corr[numvars_exist].select_dtypes(include='number').columns.tolist()
print(f"Numéricas válidas em numvars: {numvars_validas}")

# 6) resumo estatístico
if len(numvars_validas) > 0:
    display(df_corr[numvars_validas].describe())
else:
    print("Nenhuma numérica válida")

# 7) correlação
if len(numvars_validas) >= 2:
    corr = df_corr[numvars_validas].corr()

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt=".2f")
    plt.title('Correlação entre Variáveis Numéricas')
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Precisa de 2+ colunas numéricas válidas para correlação")

In [ ]:
for col in ['Icpclasse', 'Industry']:
    if col in df.columns:
        # Top 10 seguros (nlargest evita KeyError)
        top10 = df[col].value_counts().nlargest(10).index
        df_filtered = df[df[col].isin(top10)]

        plt.figure(figsize=(10,6))
        sns.countplot(data=df_filtered, x=col, order=top10, palette='muted')
        plt.title(f'Top 10 {col} (Perdidas e Ganhas)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

In [ ]:
mport matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

origem_col = 'Origem Landing Page'

if origem_col in df.columns and 'Stage' in df.columns:

    df['Stage'] = df['Stage'].astype(str)

    # separa dados
    df_perdidas = df[df['Stage'].str.contains('perdida', case=False, na=False)]
    df_ganhas = df[df['Stage'].str.contains('Cliente Cadastrado', case=False, na=False)]

    # TOP 5 baseado nas PERDIDAS
    top10_origens = (
        df_perdidas[origem_col]
        .value_counts()
        .nlargest(5)
        .index
    )

    # filtra ambos
    df_filtered = pd.concat([
        df_perdidas[df_perdidas[origem_col].isin(top10_origens)],
        df_ganhas[df_ganhas[origem_col].isin(top10_origens)]
    ])

    # ordem baseada apenas nas GANHAS
    order = (
        df_ganhas[df_ganhas[origem_col].isin(top10_origens)][origem_col]
        .value_counts()
        .index
    )

    # =========================
    # CORES / ESTILO DARK
    # =========================
    bg_color = '#0b1730'
    grid_color = '#2a3655'
    text_color = '#dfe7f5'

    plt.figure(figsize=(14, 8), facecolor=bg_color)
    ax = plt.gca()
    ax.set_facecolor(bg_color)

    sns.countplot(
        data=df_filtered,
        y=origem_col,
        hue='Stage',
        order=order,
        palette=['#d64045', '#3b82f6']  # vermelho = perdida | azul = ganha
    )

    # título e eixos
    plt.title(
        'Origens Landing Page por Stage (Perdidas vs Ganhas)',
        fontsize=16,
        color='white',
        weight='bold',
        pad=15
    )
    plt.xlabel('Quantidade de Oportunidades', color=text_color, fontsize=12)
    plt.ylabel('Origem Landing Page', color=text_color, fontsize=12)

    # ticks
    ax.tick_params(axis='x', colors=text_color, labelsize=11)
    ax.tick_params(axis='y', colors=text_color, labelsize=11)

    # grid
    ax.xaxis.grid(True, linestyle='-', alpha=0.3, color=grid_color)
    ax.yaxis.grid(False)
    ax.set_axisbelow(True)

    # bordas
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(grid_color)
    ax.spines['bottom'].set_color(grid_color)

    # legenda
    legend = plt.legend(title='Stage')
    legend.get_title().set_color('white')
    for text in legend.get_texts():
        text.set_color('white')
    legend.get_frame().set_facecolor(bg_color)
    legend.get_frame().set_edgecolor(grid_color)

    # =========================
    # VALORES NAS BARRAS
    # =========================
    for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.text(
                width + 2,
                p.get_y() + p.get_height() / 2,
                f'{int(width)}',
                va='center',
                ha='left',
                color='white',
                fontsize=9
            )

    plt.tight_layout()
    plt.show()

else:
    print("❌ 'Origem Landing Page' ou 'Stage' não encontrada")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

icp_cols = [col for col in df.columns if any(word in col.lower() for word in ['icp', 'classe', 'perfil'])]
industry_cols = [col for col in df.columns if 'industry' in col.lower() or 'setor' in col.lower()]

print("ICP-like:", icp_cols)
print("Industry-like:", industry_cols)

# Usa primeiro disponível
icp_col = icp_cols[0] if icp_cols else None
ind_col = industry_cols[0] if industry_cols else None

# =========================
# CONFIGURAÇÃO VISUAL DARK
# =========================
bg_color = '#0b1730'
grid_color = '#2a3655'
text_color = '#dfe7f5'

# =========================================================
# GRÁFICO ICP
# =========================================================
if icp_col:

    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor(bg_color)
    ax.set_facecolor(bg_color)

    df_filtered = df[df[icp_col].notna()]

    sns.countplot(
        data=df_filtered,
        x=icp_col,
        hue='Stage',
        palette=['#3b82f6', '#d64045'],
        ax=ax
    )

    ax.set_title(
        f'{icp_col} por Stage (Ganhas vs Perdidas)',
        fontsize=16,
        color='white',
        weight='bold',
        pad=15
    )

    ax.set_xlabel(icp_col, color=text_color)
    ax.set_ylabel('Quantidade', color=text_color)

    ax.tick_params(axis='x', colors=text_color, rotation=45)
    ax.tick_params(axis='y', colors=text_color)

    ax.yaxis.grid(True, linestyle='-', alpha=0.3, color=grid_color)
    ax.set_axisbelow(True)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(grid_color)
    ax.spines['bottom'].set_color(grid_color)

    legend = ax.legend(title='Stage')
    legend.get_title().set_color('white')
    for text in legend.get_texts():
        text.set_color('white')
    legend.get_frame().set_facecolor(bg_color)
    legend.get_frame().set_edgecolor(grid_color)

    # =========================
    # VALORES NAS BARRAS
    # =========================
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.text(
                p.get_x() + p.get_width()/2,
                height + 5,
                int(height),
                ha='center',
                va='bottom',
                color='white',
                fontsize=9
            )

    plt.tight_layout()
    plt.show()

# =========================================================
# GRÁFICO INDUSTRY
# =========================================================
if ind_col:

    top_ind = df[ind_col].value_counts().nlargest(5).index
    df_filt = df[df[ind_col].isin(top_ind)]

    fig, ax = plt.subplots(figsize=(14, 8))
    fig.patch.set_facecolor(bg_color)
    ax.set_facecolor(bg_color)

    sns.countplot(
        data=df_filt,
        x=ind_col,
        hue='Stage',
        palette=['#3b82f6', '#d64045'],
        order=top_ind,
        ax=ax
    )

    ax.set_title(
        f'{ind_col} por Stage',
        fontsize=16,
        color='white',
        weight='bold',
        pad=15
    )

    ax.set_xlabel(ind_col, color=text_color)
    ax.set_ylabel('Quantidade', color=text_color)

    ax.tick_params(axis='x', colors=text_color, rotation=45)
    ax.tick_params(axis='y', colors=text_color)

    ax.yaxis.grid(True, linestyle='-', alpha=0.3, color=grid_color)
    ax.set_axisbelow(True)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(grid_color)
    ax.spines['bottom'].set_color(grid_color)

    legend = ax.legend(title='Stage')
    legend.get_title().set_color('white')
    for text in legend.get_texts():
        text.set_color('white')
    legend.get_frame().set_facecolor(bg_color)
    legend.get_frame().set_edgecolor(grid_color)

    # =========================
    # VALORES NAS BARRAS
    # =========================
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.text(
                p.get_x() + p.get_width()/2,
                height + 5,
                int(height),
                ha='center',
                va='bottom',
                color='white',
                fontsize=9
            )

    plt.tight_layout()
    plt.show()

else:
    print("ℹ️ Sem colunas ICP/Industry encontradas")

In [ ]:
mport matplotlib.pyplot as plt

motivo_col = 'Motivo de Perda'

if motivo_col in df.columns:

    perdidas = df[df['Stage'].str.contains('perdida', case=False, na=False)]
    top10_motivos = perdidas[motivo_col].value_counts().nlargest(10)

    # =========================
    # CORES / ESTILO
    # =========================
    bg_color = '#0b1730'
    grid_color = '#2a3655'
    text_color = '#dfe7f5'
    bar_color = '#d64045'  # vermelho padrão "perda"

    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor(bg_color)
    ax.set_facecolor(bg_color)

    # barras
    bars = ax.barh(top10_motivos.index, top10_motivos.values, color=bar_color)

    ax.invert_yaxis()  # manter maior no topo

    # título
    ax.set_title(
        'Top 10 Principais Motivos de Perda',
        fontsize=16,
        color='white',
        weight='bold',
        pad=15
    )

    # labels
    ax.set_xlabel('Quantidade', color=text_color, fontsize=12)
    ax.set_ylabel('Motivo', color=text_color, fontsize=12)

    # grid
    ax.xaxis.grid(True, linestyle='-', alpha=0.3, color=grid_color)
    ax.set_axisbelow(True)

    # ticks
    ax.tick_params(axis='x', colors=text_color)
    ax.tick_params(axis='y', colors=text_color)

    # remover bordas
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(grid_color)
    ax.spines['bottom'].set_color(grid_color)

    # =========================
    # VALORES NAS BARRAS
    # =========================
    for bar in bars:
        width = bar.get_width()
        ax.text(
            width + 5,
            bar.get_y() + bar.get_height()/2,
            f'{int(width)}',
            va='center',
            color='white',
            fontsize=10
        )

    plt.tight_layout()
    plt.show()

else:
    print("❌ Motivo de Perda não encontrada")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Carrega os dados
data_url = "https://docs.google.com/spreadsheets/d/1hnmsJ2E4T3pV74PExZiuvLeK6cPtLOa-uhZlNfn6FNM/gviz/tq?tqx=out:csv&gid=244875212"
df = pd.read_csv(data_url)

# Filtra oportunidades perdidas
lost_df = df[df['Stage'].str.contains('Oportunidade perdida', case=False, na=False)]

# Verifica se há dados e colunas necessárias
if not lost_df.empty and 'Icp_classe' in lost_df.columns and 'Motivo de Perda' in lost_df.columns:
    # Crosstab normalizado
    ct = pd.crosstab(lost_df['Icp_classe'], lost_df['Motivo de Perda'], normalize='index')

    # Cria figura e eixo com largura maior para legenda externa
    fig, ax = plt.subplots(figsize=(14, 6))

    # Plota o gráfico
    ct.plot(kind='bar', stacked=True, ax=ax, colormap='viridis')

    # Configurações do gráfico
    ax.set_title('Motivos de Perda por Classe de ICP')
    ax.set_ylabel('Proporção')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')  # Legenda ao lado direito

    # Ajusta margem direita para dar espaço à legenda
    plt.subplots_adjust(right=0.82)

    plt.show()
else:
    print("Nenhum dado encontrado ou colunas ausentes.")

In [ ]:
ol_tempo = 'Idade da Oportunidade em Dias'
col_stage = 'Stage'

df_stage = df[
    df[col_stage].str.contains('Oportunidade perdida|Cliente Cadastrado', case=False, na=False)
].copy()

df_stage['Status'] = df_stage[col_stage].apply(
    lambda x: 'Cliente Cadastrado' if 'cliente cadastrado' in str(x).lower() else 'Oportunidade perdida'
)

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_stage,
    x='Status',
    y=col_tempo,
    order=['Cliente Cadastrado', 'Oportunidade perdida'],
    palette='magma'
)
plt.title('Tempo da oportunidade: Cliente Cadastrado x Oportunidade perdida', fontsize=16)
plt.xlabel('')
plt.ylabel('Idade da Oportunidade em Dias')
plt.tight_layout()
plt.show()

In [ ]:
ol_vendedor = 'Vendedor'
col_stage = 'Stage'

df_stage = df[
    df[col_stage].str.contains('Oportunidade perdida|Cliente Cadastrado', case=False, na=False)
].copy()

df_stage['Status'] = df_stage[col_stage].apply(
    lambda x: 'Cliente Cadastrado' if 'cliente cadastrado' in str(x).lower() else 'Oportunidade perdida'
)

# Top 10 baseado em Cliente Cadastrado
top10_vendedores = (
    df_stage[df_stage['Status'] == 'Cliente Cadastrado'][col_vendedor]
    .value_counts()
    .nlargest(10)
    .index
)

df_plot = df_stage[df_stage[col_vendedor].isin(top10_vendedores)].copy()

# Ordem também baseada em Cliente Cadastrado
ordem = (
    df_stage[df_stage['Status'] == 'Cliente Cadastrado'][col_vendedor]
    .value_counts()
    .reindex(top10_vendedores)
    .index
)

plt.figure(figsize=(14, 8))
sns.countplot(
    data=df_plot,
    y=col_vendedor,
    hue='Status',
    order=ordem,
    palette='magma'
)
plt.title('Top 10 Vendedores: Cliente Cadastrado x Oportunidade perdida', fontsize=16)
plt.xlabel('Quantidade de Oportunidades')
plt.ylabel('Vendedor')
plt.legend(title='Stage')
plt.tight_layout()
plt.show()

In [ ]:
ol_pre_vendedor = 'Pré Vendedor'
col_stage = 'Stage'

df_stage = df[
    df[col_stage].str.contains('Oportunidade perdida|Cliente Cadastrado', case=False, na=False)
].copy()

df_stage['Status'] = df_stage[col_stage].apply(
    lambda x: 'Cliente Cadastrado' if 'cliente cadastrado' in str(x).lower() else 'Oportunidade perdida'
)

# Top 10 baseado em Cliente Cadastrado
top10_pre_vendedores = (
    df_stage[df_stage['Status'] == 'Cliente Cadastrado'][col_pre_vendedor]
    .value_counts()
    .nlargest(10)
    .index
)

df_plot = df_stage[df_stage[col_pre_vendedor].isin(top10_pre_vendedores)].copy()

# Ordem também baseada em Cliente Cadastrado
ordem = (
    df_stage[df_stage['Status'] == 'Cliente Cadastrado'][col_pre_vendedor]
    .value_counts()
    .reindex(top10_pre_vendedores)
    .index
)

plt.figure(figsize=(14, 8))
sns.countplot(
    data=df_plot,
    y=col_pre_vendedor,
    hue='Status',
    order=ordem,
    palette='magma'
)
plt.title('Top 10 Pré-Vendedores: Cliente Cadastrado x Oportunidade perdida', fontsize=16)
plt.xlabel('Quantidade de Oportunidades')
plt.ylabel('Pré Vendedor')
plt.legend(title='Stage')
plt.tight_layout()
plt.show()

In [ ]:
ata_url = "https://docs.google.com/spreadsheets/d/1hnmsJ2E4T3pV74PExZiuvLeK6cPtLOa-uhZlNfn6FNM/gviz/tq?tqx=out:csv&gid=244875212"
df = pd.read_csv(data_url)

DATE_COL = 'Created Date'

df_stage = df[
    df['Stage'].astype(str).str.contains('Cliente Cadastrado|Oportunidade perdida', case=False, na=False)
].copy()

# Padronizar nomes do stage
df_stage['Stage_Group'] = df_stage['Stage'].astype(str).str.lower().map(
    lambda x: 'Ganha' if 'cliente cadastrado' in x else ('Perdida' if 'oportunidade perdida' in x else None)
)

df_stage[DATE_COL] = pd.to_datetime(df_stage[DATE_COL], errors='coerce', dayfirst=True)

# remover datas inválidas
df_stage = df_stage[df_stage[DATE_COL].notna()].copy()

df_stage = df_stage[
    ((df_stage[DATE_COL].dt.year == 2025) | (df_stage[DATE_COL].dt.year == 2026)) &
    (df_stage[DATE_COL].dt.month.isin([1, 2]))
].copy()

# Criar rótulo de mês/ano
df_stage['Mes_Ano'] = df_stage[DATE_COL].dt.strftime('%m/%Y')

# Ordem correta no eixo X
ordem_meses = ['01/2025', '02/2025', '01/2026', '02/2026']

plt.figure(figsize=(10, 5))
sns.countplot(
    data=df_stage,
    x='Mes_Ano',
    hue='Stage_Group',
    order=ordem_meses
)

plt.title('Quantidade de Oportunidades Ganhas x Perdidas por Mês')
plt.xlabel('Mês/Ano')
plt.ylabel('Quantidade')
plt.legend(title='Resultado')
plt.tight_layout()
plt.show()

In [ ]:
mport pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Corrected data_url for CSV export
data_url = "https://docs.google.com/spreadsheets/d/1hnmsJ2E4T3pV74PExZiuvLeK6cPtLOa-uhZlNfn6FNM/export?format=csv&gid=946629782"
df = pd.read_csv(data_url)

cols = ["Amount", "Idade da Oportunidade em Dias"]
df_model = df[cols].copy()

for c in cols:
    if df_model[c].dtype == 'object':
        df_model[c] = df_model[c].astype(str).str.replace(',', '.', regex=False)
    df_model[c] = pd.to_numeric(df_model[c], errors="coerce")

# Remover nulos
df_model = df_model.dropna().reset_index(drop=True)

# -------------------------
# 3) Padronização
# -------------------------
scaler = StandardScaler()
X = scaler.fit_transform(df_model)

# -------------------------
# 4) Elbow e Silhouette
# -------------------------
sse = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    sse.append(model.inertia_)
    silhouettes.append(silhouette_score(X, labels))

# -------------------------
# 5) Gráficos de avaliação
# -------------------------
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(list(K_range), sse, marker="o")
ax[0].set_title("Método Elbow")
ax[0].set_xlabel("Número de clusters")
ax[0].set_ylabel("SSE")

ax[1].plot(list(K_range), silhouettes, marker="o", color="green")
ax[1].set_title("Método Silhouette")
ax[1].set_xlabel("Número de clusters")
ax[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

# Escolha do melhor K pelo maior silhouette
best_k = list(K_range)[int(np.argmax(silhouettes))]
print("Melhor K sugerido:", best_k)

# -------------------------
# 6) Modelo final
# -------------------------
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_model["cluster"] = kmeans.fit_predict(X)

# -------------------------
# 7) Resumo dos clusters
# -------------------------
summary = df_model.groupby("cluster")[cols].agg(["count", "mean", "median", "min", "max"]).round(2)
print(summary)

# Percentual de registros por cluster
pct = (df_model["cluster"].value_counts(normalize=True).sort_index() * 100).round(2)
print("\nPercentual por cluster:")
print(pct)

# -------------------------
# 8) Visualizações
# -------------------------
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_model,
    x="Idade da Oportunidade em Dias",
    y="Amount",
    hue="cluster",
    palette="tab10",
    s=70
)
plt.title("Clusters K-Means")
plt.tight_layout()
plt.show()

# -------------------------
# 9) Exportação
# -------------------------
df_saida = df.copy()
df_saida["cluster"] = df_model["cluster"]

df_saida.to_csv("kmeans_resultado.csv", index=False)
print("Arquivo salvo: kmeans_resultado.csv")

print("\nInterpretação inicial dos clusters:")

for cluster_id in sorted(df_model["cluster"].unique()):
    media_amount = df_model[df_model["cluster"] == cluster_id]["Amount"].mean()
    media_idade = df_model[df_model["cluster"] == cluster_id]["Idade da Oportunidade em Dias"].mean()
    qtd = (df_model["cluster"] == cluster_id).sum()

    print(f"\nCluster {cluster_id}:")
    print(f"- Quantidade de registros: {qtd}")
    print(f"- Amount médio: {media_amount:.2f}")
    print(f"- Idade média da oportunidade: {media_idade:.2f}")

centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

centroids_df = pd.DataFrame(centroids_original, columns=cols)
centroids_df["cluster"] = range(best_k)

print("\nCentróides na escala original:")
display(centroids_df)

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_model,
    x="Idade da Oportunidade em Dias",
    y="Amount",
    hue="cluster",
    palette="tab10",
    s=70,
    alpha=0.7
)

plt.scatter(
    centroids_df["Idade da Oportunidade em Dias"],
    centroids_df["Amount"],
    c="black",
    s=250,
    marker="X",
    label="Centróides"
)

plt.title("Clusters K-Means com Centróides")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))
sns.countplot(data=df_model, x="cluster", palette="tab10")
plt.title("Quantidade de Registros por Cluster")
plt.xlabel("Cluster")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))
sns.boxplot(data=df_model, x="cluster", y="Amount", palette="tab10")
plt.title("Distribuição de Amount por Cluster")
plt.xlabel("Cluster")
plt.ylabel("Amount")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_model, x="cluster", y="Idade da Oportunidade em Dias", palette="tab10")
plt.title("Distribuição da Idade da Oportunidade por Cluster")
plt.xlabel("Cluster")
plt.ylabel("Idade da Oportunidade em Dias")
plt.tight_layout()
plt.show()

cluster_profile = df_model.groupby("cluster")[cols].mean().round(2)
print("\nPerfil médio dos clusters:")
display(cluster_profile)

ranking_amount = (
    df_model.groupby("cluster")["Amount"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("\nRanking dos clusters por Amount médio:")
print(ranking_amount)

centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

centroids_df = pd.DataFrame(centroids_original, columns=cols)
centroids_df["cluster"] = range(best_k)

print("\nCentróides na escala original:")
display(centroids_df)

In [ ]:
Último código (refatorado)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

# =========================
# 1) CARREGAR BASE
# =========================
data_url = "https://docs.google.com/spreadsheets/d/1hnmsJ2E4T3pV74PExZiuvLeK6cPtLOa-uhZlNfn6FNM/export?format=csv&gid=244875212"
df = pd.read_csv(data_url)

# =========================
# 2) TARGET
# =========================
df["target"] = np.where(
    df["Stage"].astype(str).str.lower().str.contains("cliente cadastrado", na=False),
    1,
    np.where(
        df["Stage"].astype(str).str.lower().str.contains("oportunidade perdida", na=False),
        0,
        np.nan
    )
)

df_model = df[df["target"].notna()].copy()
df_model["target"] = df_model["target"].astype(int)

# =========================
# 3) TRATAR AMOUNT
# =========================
if "Amount" in df_model.columns:
    df_model["Amount"] = (
        df_model["Amount"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df_model["Amount"] = pd.to_numeric(df_model["Amount"], errors="coerce")

# =========================
# 4) FEATURES
# =========================
candidate_features = [
    "Amount",
    "Idade da Oportunidade em Dias",
    "Icp_classe",
    "Industry",
    "Origem Landing Page"
]

features = [col for col in candidate_features if col in df_model.columns]

X = df_model[features].copy()
y = df_model["target"].copy()

numeric_features = [col for col in features if pd.api.types.is_numeric_dtype(X[col])]
categorical_features = [col for col in features if col not in numeric_features]

# =========================
# 5) PREPROCESSAMENTO
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# =========================
# 6) SPLIT TEMPORAL
# =========================
df_model["Created Date"] = pd.to_datetime(
    df_model["Created Date"],
    dayfirst=True,
    errors="coerce"
)

df_model = df_model.dropna(subset=["Created Date"])
df_model = df_model.sort_values("Created Date")

X = df_model[features]
y = df_model["target"]

split_index = int(len(df_model) * 0.7)

X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]

X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

# =========================
# 7) MODELO
# =========================
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=3,
        min_samples_split=50,
        min_samples_leaf=30,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# =========================
# 8) PREDIÇÕES
# =========================
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# =========================
# 9) FEATURE NAMES
# =========================
ohe = model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]

cat_names = ohe.get_feature_names_out(categorical_features) if len(categorical_features) > 0 else np.array([])
feature_names = np.concatenate([numeric_features, cat_names])

# =========================
# 10) FEATURE IMPORTANCE
# =========================
tree_clf = model.named_steps["classifier"]

importances = tree_clf.feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(feat_imp.head(10))

plt.figure()
sns.barplot(
    data=feat_imp.head(10),
    x="importance",
    y="feature"
)
plt.title("Feature Importance")
plt.show()

# =========================
# 11) SHAP
# =========================
X_train_transformed = model.named_steps["preprocessor"].transform(X_train)
X_test_transformed = model.named_steps["preprocessor"].transform(X_test)

if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()
X_test_transformed = X_test_transformed.astype(float)

explainer = shap.TreeExplainer(tree_clf)
shap_values = explainer.shap_values(X_test_transformed)

shap.summary_plot(
    shap_values[:, :, 1],
    X_test_transformed,
    feature_names=feature_names
)

# =========================
# 12) MÉTRICAS
# =========================
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_proba)

print("\n=== MÉTRICAS DO MODELO ===")
print(f"Acurácia : {acc:.4f}")
print(f"Precisão : {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=["Perdida", "Ganha"], zero_division=0))

# =========================
# 13) MATRIZ DE CONFUSÃO
# =========================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred. Perdida", "Pred. Ganha"],
    yticklabels=["Real Perdida", "Real Ganha"]
)
plt.title("Matriz de Confusão")
plt.xlabel("Predição")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

# =========================
# 14) ROC CURVE
# =========================
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

# =========================
# 15) ÁRVORE
# =========================
plt.figure(figsize=(22, 10))
plot_tree(
    tree_clf,
    feature_names=feature_names,
    class_names=["Perdida", "Ganha"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree")
plt.show()